# MVP07 — Camada Gold

Preenche as tabelas do modelo dimensional **criadas vazias no MVP03-modelagem**, a partir da Silver.

**Ordem de carga:** dimensões primeiro, fatos depois.

| Tabela | Grão | Origem |
|---|---|---|
| `dim_tempo` | um dia | gerada por sequência de datas |
| `dim_cliente` | um customer_id | silver.clientes |
| `dim_vendedor` | um vendedor | silver.vendedores |
| `dim_produto` | um produto | silver.produtos |
| `fato_pedidos` | um pedido | silver.pedidos + itens_pedido + pagamentos + avaliacoes |
| `fato_itens_pedido` | um item de pedido | silver.itens_pedido + pedidos + clientes + vendedores |

## Tabela de referência UF → Região

View temporária usada para enriquecer as dimensões de cliente e vendedor.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW uf_regiao AS
SELECT * FROM VALUES
  ('AC','Norte'), ('AM','Norte'), ('AP','Norte'), ('PA','Norte'), ('RO','Norte'), ('RR','Norte'), ('TO','Norte'),
  ('AL','Nordeste'), ('BA','Nordeste'), ('CE','Nordeste'), ('MA','Nordeste'), ('PB','Nordeste'),
  ('PE','Nordeste'), ('PI','Nordeste'), ('RN','Nordeste'), ('SE','Nordeste'),
  ('DF','Centro-Oeste'), ('GO','Centro-Oeste'), ('MS','Centro-Oeste'), ('MT','Centro-Oeste'),
  ('ES','Sudeste'), ('MG','Sudeste'), ('RJ','Sudeste'), ('SP','Sudeste'),
  ('PR','Sul'), ('RS','Sul'), ('SC','Sul')
AS t(uf, regiao);

SELECT COUNT(*) AS ufs_mapeadas FROM uf_regiao;

## dim_tempo

Gerada com `sequence()` do primeiro dia do mês da primeira compra até o último dia do mês da última compra. Nomes de mês e dia em português via `element_at`.

In [0]:
%sql
INSERT OVERWRITE mvp.gold.dim_tempo
SELECT
  d                                                        AS data,
  year(d)                                                  AS ano,
  quarter(d)                                               AS trimestre,
  month(d)                                                 AS mes,
  element_at(array('janeiro','fevereiro','março','abril','maio','junho','julho',
                   'agosto','setembro','outubro','novembro','dezembro'), month(d)) AS nome_mes,
  date_format(d, 'yyyy-MM')                                AS ano_mes,
  dayofweek(d)                                             AS dia_semana,
  element_at(array('domingo','segunda-feira','terça-feira','quarta-feira',
                   'quinta-feira','sexta-feira','sábado'), dayofweek(d))           AS nome_dia_semana,
  dayofweek(d) IN (1, 7)                                   AS flag_fim_semana
FROM (
  SELECT explode(sequence(ini, fim, INTERVAL 1 DAY)) AS d
  FROM (
    SELECT trunc(MIN(CAST(data_hora_compra AS DATE)), 'MM') AS ini,
           last_day(MAX(CAST(data_hora_compra AS DATE)))   AS fim
    FROM mvp.silver.pedidos
  ) limites
) datas;

## dim_cliente e dim_vendedor

In [0]:
%sql
INSERT OVERWRITE mvp.gold.dim_cliente
SELECT c.customer_id, c.customer_unique_id, c.cep_prefixo, c.cidade, c.uf, r.regiao
FROM mvp.silver.clientes c
LEFT JOIN uf_regiao r ON c.uf = r.uf;

In [0]:
%sql
INSERT OVERWRITE mvp.gold.dim_vendedor
SELECT v.seller_id, v.cep_prefixo, v.cidade, v.uf, r.regiao
FROM mvp.silver.vendedores v
LEFT JOIN uf_regiao r ON v.uf = r.uf;

## dim_produto

`volume_cm3` só é calculado quando as três medidas existem; caso contrário fica nulo.

In [0]:
%sql
INSERT OVERWRITE mvp.gold.dim_produto
SELECT
  product_id,
  categoria,
  categoria_en,
  peso_g,
  comprimento_cm,
  altura_cm,
  largura_cm,
  CASE WHEN comprimento_cm IS NOT NULL AND altura_cm IS NOT NULL AND largura_cm IS NOT NULL
       THEN comprimento_cm * altura_cm * largura_cm END AS volume_cm3,
  qtd_fotos
FROM mvp.silver.produtos;

## fato_pedidos

Consolida no grão de pedido três agregações independentes, unidas por `LEFT JOIN` para que nenhum pedido se perca:

- **Itens:** quantidade de itens e de vendedores, soma de preços e fretes, e `flag_interestadual`.
- **Pagamentos:** valor pago total, maior número de parcelas e meio principal.
- **Avaliação:** já deduplicada na Silver, uma por pedido.

Métricas de prazo (`dias_entrega`, `dias_atraso`, `flag_atraso`) só são calculadas quando `flag_entrega_valida` é verdadeiro; nos demais pedidos ficam nulas.

Pedidos sem itens permanecem na fato com valores nulos e são excluídos das análises de receita.

In [0]:
%sql
INSERT OVERWRITE mvp.gold.fato_pedidos
SELECT
  p.order_id,
  p.customer_id,
  CAST(p.data_hora_compra AS DATE)                               AS data_compra,
  p.status_pedido,
  p.data_hora_compra,
  p.data_entrega,
  p.data_estimada,
  CAST(i.qtd_itens AS INT)                                       AS qtd_itens,
  CAST(i.qtd_vendedores AS INT)                                  AS qtd_vendedores,
  CAST(i.valor_produtos AS DECIMAL(12,2))                        AS valor_produtos,
  CAST(i.valor_frete AS DECIMAL(12,2))                           AS valor_frete,
  CAST(i.valor_produtos + i.valor_frete AS DECIMAL(12,2))        AS valor_total,
  CAST(pg.valor_pago AS DECIMAL(12,2))                           AS valor_pago,
  pg.tipo_pagamento_principal,
  CAST(pg.qtd_parcelas AS INT)                                   AS qtd_parcelas,
  CASE WHEN p.flag_entrega_valida
       THEN datediff(p.data_entrega, p.data_hora_compra) END     AS dias_entrega,
  CASE WHEN p.flag_entrega_valida
       THEN datediff(p.data_entrega, p.data_estimada) END        AS dias_atraso,
  CASE WHEN p.flag_entrega_valida
       THEN datediff(p.data_entrega, p.data_estimada) > 0 END    AS flag_atraso,
  i.flag_interestadual,
  a.nota                                                         AS nota_avaliacao
FROM mvp.silver.pedidos p
LEFT JOIN (
  SELECT it.order_id,
         COUNT(*)                                                AS qtd_itens,
         COUNT(DISTINCT it.seller_id)                            AS qtd_vendedores,
         SUM(it.preco)                                           AS valor_produtos,
         SUM(it.valor_frete)                                     AS valor_frete,
         MAX(CASE WHEN v.uf <> c.uf THEN 1 ELSE 0 END) = 1       AS flag_interestadual
  FROM mvp.silver.itens_pedido it
  JOIN mvp.silver.pedidos pp         ON it.order_id    = pp.order_id
  LEFT JOIN mvp.silver.vendedores v  ON it.seller_id   = v.seller_id
  LEFT JOIN mvp.silver.clientes c    ON pp.customer_id = c.customer_id
  GROUP BY it.order_id
) i  ON p.order_id = i.order_id
LEFT JOIN (
  SELECT order_id,
         SUM(valor_pagamento)                                    AS valor_pago,
         MAX(parcelas)                                           AS qtd_parcelas,
         max_by(tipo_pagamento,
                CASE WHEN tipo_pagamento <> 'not_defined' THEN valor_pagamento END) AS tipo_pagamento_principal
  FROM mvp.silver.pagamentos
  GROUP BY order_id
) pg ON p.order_id = pg.order_id
LEFT JOIN mvp.silver.avaliacoes a ON p.order_id = a.order_id;

## fato_itens_pedido

In [0]:
%sql
INSERT OVERWRITE mvp.gold.fato_itens_pedido
SELECT
  it.order_id,
  it.order_item_id,
  it.product_id,
  it.seller_id,
  p.customer_id,
  CAST(p.data_hora_compra AS DATE) AS data_compra,
  it.preco,
  it.valor_frete,
  v.uf <> c.uf                     AS flag_interestadual
FROM mvp.silver.itens_pedido it
JOIN mvp.silver.pedidos p         ON it.order_id   = p.order_id
LEFT JOIN mvp.silver.vendedores v ON it.seller_id  = v.seller_id
LEFT JOIN mvp.silver.clientes c   ON p.customer_id = c.customer_id;

## Validação 1 — Volumes por tabela

Esperado: 
- `fato_pedidos` = linhas de `silver.pedidos`
- `fato_itens_pedido` = linhas de `silver.itens_pedido`


In [0]:
%sql
SELECT 'dim_tempo'         AS tabela, COUNT(*) AS linhas FROM mvp.gold.dim_tempo         UNION ALL
SELECT 'dim_cliente',                 COUNT(*)           FROM mvp.gold.dim_cliente       UNION ALL
SELECT 'dim_vendedor',                COUNT(*)           FROM mvp.gold.dim_vendedor      UNION ALL
SELECT 'dim_produto',                 COUNT(*)           FROM mvp.gold.dim_produto       UNION ALL
SELECT 'fato_pedidos',                COUNT(*)           FROM mvp.gold.fato_pedidos      UNION ALL
SELECT 'fato_itens_pedido',           COUNT(*)           FROM mvp.gold.fato_itens_pedido;

## Validação 2 — Unicidade das chaves primárias

As chaves no Unity Catalog são informativas, se espera zero em todas as linhas.

In [0]:
%sql
SELECT 'dim_tempo' AS tabela,    COUNT(*) - COUNT(DISTINCT data)                    AS duplicadas FROM mvp.gold.dim_tempo    UNION ALL
SELECT 'dim_cliente',            COUNT(*) - COUNT(DISTINCT customer_id)                           FROM mvp.gold.dim_cliente  UNION ALL
SELECT 'dim_vendedor',           COUNT(*) - COUNT(DISTINCT seller_id)                             FROM mvp.gold.dim_vendedor UNION ALL
SELECT 'dim_produto',            COUNT(*) - COUNT(DISTINCT product_id)                            FROM mvp.gold.dim_produto  UNION ALL
SELECT 'fato_pedidos',           COUNT(*) - COUNT(DISTINCT order_id)                              FROM mvp.gold.fato_pedidos UNION ALL
SELECT 'fato_itens_pedido',      COUNT(*) - COUNT(DISTINCT order_id, order_item_id)               FROM mvp.gold.fato_itens_pedido;

## Validação 3 — Integridade referencial (fatos to dimensões)

Pela mesma razão, as chaves estrangeiras são verificadas explicitamente.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM mvp.gold.fato_pedidos f      LEFT ANTI JOIN mvp.gold.dim_cliente d  ON f.customer_id = d.customer_id) AS pedidos_sem_cliente,
  (SELECT COUNT(*) FROM mvp.gold.fato_pedidos f      LEFT ANTI JOIN mvp.gold.dim_tempo d    ON f.data_compra = d.data)        AS pedidos_sem_data,
  (SELECT COUNT(*) FROM mvp.gold.fato_itens_pedido f LEFT ANTI JOIN mvp.gold.dim_produto d  ON f.product_id  = d.product_id)  AS itens_sem_produto,
  (SELECT COUNT(*) FROM mvp.gold.fato_itens_pedido f LEFT ANTI JOIN mvp.gold.dim_vendedor d ON f.seller_id   = d.seller_id)   AS itens_sem_vendedor,
  (SELECT COUNT(*) FROM mvp.gold.fato_itens_pedido f LEFT ANTI JOIN mvp.gold.dim_cliente d  ON f.customer_id = d.customer_id) AS itens_sem_cliente,
  (SELECT COUNT(*) FROM mvp.gold.fato_itens_pedido f LEFT ANTI JOIN mvp.gold.dim_tempo d    ON f.data_compra = d.data)        AS itens_sem_data;

## Validação 4 — Reconciliação de valores

A receita deve ser idêntica nas três visões. Qualquer diferença indicaria itens perdidos ou duplicados nos joins.

In [0]:
%sql
SELECT
  (SELECT SUM(preco)          FROM mvp.silver.itens_pedido)    AS receita_silver,
  (SELECT SUM(preco)          FROM mvp.gold.fato_itens_pedido) AS receita_fato_itens,
  (SELECT SUM(valor_produtos) FROM mvp.gold.fato_pedidos)      AS receita_fato_pedidos;

Os 3 valores são iguais, portanto nada é duplicado ou perdido nas joins.

## Validação 5 — Domínios observados

Mínimos e máximos reais das métricas derivadas.

In [0]:
%sql
SELECT
  MIN(nota_avaliacao) AS nota_min,     MAX(nota_avaliacao) AS nota_max,
  MIN(dias_entrega)   AS entrega_min,  MAX(dias_entrega)   AS entrega_max,
  MIN(dias_atraso)    AS atraso_min,   MAX(dias_atraso)    AS atraso_max,
  MIN(qtd_itens)      AS itens_min,    MAX(qtd_itens)      AS itens_max,
  MIN(qtd_parcelas)   AS parcelas_min, MAX(qtd_parcelas)   AS parcelas_max,
  MIN(valor_total)    AS valor_min,    MAX(valor_total)    AS valor_max,
  MIN(data_compra)    AS primeira_compra, MAX(data_compra) AS ultima_compra
FROM mvp.gold.fato_pedidos;

## Resultado

Modelo estrela materializado em `mvp.gold`, com chaves únicas, integridade referencial verificada e receita
reconciliada com a Silver.